# Arabic Cursive OCR Benchmark — Local Models on Colab

Runs open-weight / Arabic-specific VLMs (Qari-OCR, Qwen2.5-VL, PaliGemma-Arabic)
against the **Muharaf** handwritten-Arabic test set and reports CER / WER /
diacritic-F1 / latency.

**Before running:** `Runtime -> Change runtime type -> T4 GPU`, and add a
`GH_TOKEN` secret (see `notebooks/README.md`). ~10-20 min end to end.

## 1. Check the GPU

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> T4 GPU'
print(torch.cuda.get_device_name(0))
!nvidia-smi --query-gpu=memory.total,memory.used --format=csv

## 2. Clone the benchmark repo

In [ ]:
# Clone the (private) benchmark repo using a fine-grained PAT stored as a
# Colab secret named GH_TOKEN.  See notebooks/README.md to create one.
import os, subprocess
GH_USER, REPO = 'rabeeh97', 'arabic-ocr-benchmark'
token = None
try:
    from google.colab import userdata
    token = userdata.get('GH_TOKEN')
except Exception as e:
    print('no GH_TOKEN secret:', e)
if not os.path.isdir(REPO):
    auth = f'{token}@' if token else ''
    url = f'https://{auth}github.com/{GH_USER}/{REPO}.git'
    subprocess.run(['git', 'clone', '--depth', '1', url], check=True)
os.chdir(REPO)
print(subprocess.run(['git','log','--oneline','-1'], capture_output=True, text=True).stdout)

## 3. Install the harness (with local-model deps)

In [ ]:
# Keep all downloads on Colab's own scratch disk (ephemeral, ~78 GB) --
# never the local machine. Nothing here touches your computer.
import os
os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
# Optional HF_TOKEN secret: lifts anonymous rate limits and speeds the
# Muharaf download (and is required for gated models like PaliGemma's base).
try:
    from google.colab import userdata
    _hf = userdata.get('HF_TOKEN')
    if _hf:
        os.environ['HF_TOKEN'] = _hf
        print('HF_TOKEN set')
except Exception:
    pass

In [ ]:
!pip install -q -e '.[local]'
# Colab ships a recent torch already; this pulls transformers>=4.49, peft, qwen-vl-utils, ...

## 4. Pick the model(s)

One name for a single run, or a list for a **sweep** — the runner loads
each model, runs every line, frees it, then loads the next, so a list fits
on one T4. Names are in `configs/models.yaml`:
`qari-ocr-0.3`, `qari-ocr-0.2`, `qwen2.5-vl-3b` (all fp16-safe on a free T4),
`qwen2.5-vl-7b` (needs 4-bit), `paligemma-arabic-hw` (gated + line-only).
The list below mirrors `configs/run.sweep-local.yaml`.

In [ ]:
MODELS = ['qari-ocr-0.3']            # e.g. ['qari-ocr-0.3','qari-ocr-0.2','qwen2.5-vl-3b']
LIMIT  = 50                          # Muharaf test lines; None = all 1,334

run_cfg = {
    'models': MODELS,
    'datasets': ['muharaf_lines'],
    'limit': LIMIT,
    'prompt_version': 'line-v2',
    'normalization_profiles': ['raw', 'light', 'diac_insensitive'],
    'concurrency': 1,
    'results_dir': 'results',
}

## 5. Sanity check on one image

Loads the model and transcribes a single line — surfaces any load / prompt /
dtype problem in ~1 min instead of after the whole run.

In [ ]:
from arabic_ocr_bench.models.registry import build_model, load_model_configs
from arabic_ocr_bench.datasets.registry import build_dataset, load_dataset_configs
from arabic_ocr_bench.prompting import get_prompt

_m = load_model_configs(); _d = load_dataset_configs()
_client = build_model(MODELS[0], _m[MODELS[0]])
sample = next(iter(build_dataset('muharaf_lines', _d['muharaf_lines'], limit=1)))
pred = _client.transcribe(sample.image, get_prompt('line-v2'))
display(sample.image)
print('MODEL:', MODELS[0])
print('GT   :', sample.ground_truth)
print('PRED :', pred.cleaned_text or f'(empty / error: {pred.error})')
if hasattr(_client, 'close'): _client.close()   # free VRAM before the real run
assert pred.error is None, pred.error

## 6. Run the benchmark

In [ ]:
from arabic_ocr_bench.runner import run
from arabic_ocr_bench.report import score_run

run_dir = run(run_cfg)
summary_md = score_run(run_dir, profiles=tuple(run_cfg['normalization_profiles']))
print(open(summary_md).read())

## 7. Inspect predictions

In [ ]:
import json, pandas as pd, pathlib
rows = [json.loads(l) for l in (run_dir / 'predictions.jsonl').read_text().splitlines()]
df = pd.DataFrame(rows)[['model','sample_id','ground_truth','cleaned_text','latency_ms','error']]
pd.set_option('display.max_colwidth', 60)
df.head(20)

## 8. Leaderboard

In [ ]:
# Build the shareable HTML leaderboard from every run under results/
!python scripts/make_leaderboard.py results -o results/leaderboard.html
from IPython.display import HTML
HTML(open('results/leaderboard.html').read())

## 9. Save results to Google Drive

In [ ]:
# Persist this run's results to Google Drive (they vanish when Colab recycles).
from google.colab import drive
drive.mount('/content/drive')
import shutil, pathlib
dst = pathlib.Path('/content/drive/MyDrive/arabic-ocr-bench/results') / run_dir.name
dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(run_dir, dst, dirs_exist_ok=True)
print('saved ->', dst)

## 10. Commit results back to the repo

So they show up here and on GitHub without a manual download.

In [ ]:
# Push this run's results back to the repo (needs the GH_TOKEN secret used to
# clone). Skips silently if no token. Commits results/<run_id>/ + leaderboard.
import subprocess, os
if token:
    subprocess.run(['git','config','user.email','colab@arabic-ocr-bench'], check=True)
    subprocess.run(['git','config','user.name','colab-runner'], check=True)
    subprocess.run(['git','add','-f', f'results/{run_dir.name}', 'results/leaderboard.html', 'results/leaderboard.json'], check=True)
    msg = f'results: {", ".join(run_cfg["models"])} x {", ".join(run_cfg["datasets"])} (n={run_cfg.get("limit") or "all"})'
    r = subprocess.run(['git','commit','-m',msg], capture_output=True, text=True)
    print(r.stdout or r.stderr)
    push = subprocess.run(['git','push','origin','HEAD:main'], capture_output=True, text=True)
    print(push.stdout or push.stderr)
else:
    print('no GH_TOKEN -- results are on Drive only; download them from the file browser')

## Next

- **Sweep**: set `MODELS = ['qari-ocr-0.3','qari-ocr-0.2','qwen2.5-vl-3b']` and re-run cells 4-11. One `results/<run_id>/` holds all of them; the leaderboard shows a row each.
- Set `LIMIT = None` for the full 1,334-line test split.
- Hosted models (Gemini/GPT/Claude) → `02_colab_hosted_apis.ipynb`, same dataset & metrics.
- `03_analysis.ipynb` — worst-error browser (image + GT + each model's output).
- Full walkthrough incl. publishing the artifact: `docs/first-benchmark-runbook.md`.